In [1]:
import os
import sys
import json
import time
import signal
import subprocess
from pathlib import Path

REPO = "https://github.com/eslam-ahmed43/llm-lab.git"
ROOT = Path("/kaggle/working/llm-lab")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
(ROOT / "logs").mkdir(exist_ok=True)
(ROOT / "plots").mkdir(exist_ok=True)
for name in ["src/exp7_launch_model.py", "src/exp7_fit.py", "PREREGISTRATION_EXP7B.md"]:
    assert (ROOT / name).exists(), f"{name} is missing: push it to GitHub first"
assert "--dtype" in (ROOT / "src/exp7_launch_model.py").read_text(), "the updated measurement script is not on GitHub yet"

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("git log --oneline -3; ls src", shell=True, capture_output=True, text=True).stdout)
print(sys.version)

Cloning into '/kaggle/working/llm-lab'...


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

fa55edc Exp 7b: fp32 option, dtype-aware fit, acceptance criteria
435a3c9 Preregister Exp 7b predictions before measuring
08d280e Exp 7 results: launch-aware decode model, fp16, three sizes
bench_inference.py
exp5_internals.py
exp6_profiling.py
exp7_fit.py
exp7_launch_model.py
hf_batched_server.py
hf_server.py
plot_exp6.py
plot_lora.py
quantize.py
train_lora.py

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
%%bash
set -e
python -c "
from huggingface_hub import snapshot_download
for m in ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct', 'Qwen/Qwen2.5-3B-Instruct']:
    snapshot_download(m)
    print('cached', m)
"

cached Qwen/Qwen2.5-0.5B-Instruct
cached Qwen/Qwen2.5-1.5B-Instruct
cached Qwen/Qwen2.5-3B-Instruct


Fetching 12 files: 100%|██████████| 12/12 [00:34<00:00,  2.85s/it]


In [3]:
def run_script(script, tag, args, timeout_s=1800):
    log_path = ROOT / "logs" / f"exp7_{tag}.log"
    cmd = [sys.executable, "-X", "faulthandler", "-u", str(ROOT / "src" / script)] + args
    env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}

    def new_lines(shown):
        lines = log_path.read_text(errors="replace").splitlines()
        for line in lines[shown:]:
            if line.strip() and "Loading weights" not in line and "unauthenticated" not in line:
                print(line[:400], flush=True)
        return len(lines)

    start, shown = time.time(), 0
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(ROOT))
        while proc.poll() is None:
            time.sleep(5)
            shown = new_lines(shown)
            if time.time() - start > timeout_s:
                os.kill(proc.pid, signal.SIGABRT)
                time.sleep(3)
                proc.kill()
                print(log_path.read_text(errors="replace")[-3000:])
                raise TimeoutError(f"{tag} exceeded {timeout_s}s")
    new_lines(shown)
    if proc.returncode != 0:
        print(log_path.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"{tag} failed with exit code {proc.returncode}")

In [4]:
import traceback

runs = [
    ("1p5b_fp16", "Qwen/Qwen2.5-1.5B-Instruct", "fp16"),
    ("0p5b_fp32", "Qwen/Qwen2.5-0.5B-Instruct", "fp32"),
    ("1p5b_fp32", "Qwen/Qwen2.5-1.5B-Instruct", "fp32"),
    ("3b_fp32", "Qwen/Qwen2.5-3B-Instruct", "fp32"),
]
for tag, model, dtype in runs:
    out = ROOT / "results" / f"launch7b_{tag}.json"
    if out.exists():
        print(tag, "already done")
        continue
    try:
        run_script("exp7_launch_model.py", f"7b_{tag}",
                   ["--model", model, "--dtype", dtype, "--out", str(out)], timeout_s=3000)
    except Exception:
        traceback.print_exc()

[fp16] building Qwen/Qwen2.5-1.5B-Instruct
  1.54B parameters, 28 layers, weights 3.09 GB, bandwidth 242 GB/s, weight-streaming floor 12.7 ms
  dynamic_eager_nosync: 25.95 ms/step (enqueue alone 25.93 ms)
  static_eager_nosync: 27.10 ms/step (enqueue alone 27.08 ms)
  static_cuda_graph: 19.70 ms/step (enqueue alone 18.80 ms)
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
  profile dynamic_eager_nosync: 1312 device operations per step, GPU busy 17.04 ms
  profile static_eager_nosync: 1398 device operations per step, GPU busy 18.93 ms
  profile static_cuda_graph: 1400 device operations per step, GPU busy 20.73 ms
saved /kaggle/working/llm-lab/results/launch7b_1p5b_fp16.json
[fp32] building Qwen/Qwen2.5-0.5B-Instruct
  0.49B parameters, 24 layers, weights 1.98 GB, bandwidth 242 GB

In [5]:
run_script("exp7_fit.py", "7b_fit",
           ["--results-dir", str(ROOT / "results"), "--prefix", "launch7b", "--reference", "1.5B",
            "--reference-dtype", "fp16", "--out", str(ROOT / "results/launch7b_fit.json"),
            "--plot", str(ROOT / "plots/launch7b_model.png")])

fitted on Qwen2.5-1.5B-Instruct fp16: c_layer 248 us, 49.9 operations per layer, t_launch 19.4 us per operation
run                            meas eager    pred   err%   meas graph    pred   err%   speedup m/p  regime meas / pred
Qwen2.5-0.5B-Instruct fp32          20.38   23.23  +14.0        14.24   14.11   -0.9   1.43 / 1.65  CPU-bound / CPU-bound
Qwen2.5-1.5B-Instruct fp16          27.10   27.10   +0.0        19.70   19.70   +0.0   1.38 / 1.38  CPU-bound / CPU-bound  (fit)
Qwen2.5-1.5B-Instruct fp32          32.12   32.44   +1.0        32.59   32.44   -0.4   0.99 / 1.00  CPU-bound / GPU-bound
Qwen2.5-3B-Instruct fp32            59.88   59.90   +0.0        59.84   59.90   +0.1   1.00 / 1.00  CPU-bound / GPU-bound
mean absolute error over all predicted eager and graph steps of the test runs: model 2.7%, bandwidth_only 38.1%, layers_only 31.2%
regimes measured in the test runs: Qwen2.5-0.5B-Instruct fp32: CPU-bound, Qwen2.5-1.5B-Instruct fp32: CPU-bound, Qwen2.5-3B-Instruct fp32: CPU-

In [6]:
import zipfile

with zipfile.ZipFile("/kaggle/working/llm-lab-exp7b.zip", "w", zipfile.ZIP_DEFLATED) as z:
    files = sorted((ROOT / "results").glob("launch7b_*.json"))
    files += sorted((ROOT / "plots").glob("launch7b_model.png")) + sorted((ROOT / "logs").glob("exp7_7b_*.log"))
    for f in files:
        z.write(f, arcname=f"{f.parent.name}/{f.name}")
print("Zipped:", zipfile.ZipFile("/kaggle/working/llm-lab-exp7b.zip").namelist())

Zipped: ['results/launch7b_0p5b_fp32.json', 'results/launch7b_1p5b_fp16.json', 'results/launch7b_1p5b_fp32.json', 'results/launch7b_3b_fp32.json', 'results/launch7b_fit.json', 'plots/launch7b_model.png', 'logs/exp7_7b_0p5b_fp32.log', 'logs/exp7_7b_1p5b_fp16.log', 'logs/exp7_7b_1p5b_fp32.log', 'logs/exp7_7b_3b_fp32.log', 'logs/exp7_7b_fit.log']
